# 03 -- BRFSS 2022 (Behavioral Risk Factor Surveillance System)

## Research question

Same as datasets 1 & 2: how much of an observed subgroup performance gap is
genuine algorithmic bias versus an artifact of prevalence, calibration, and
case mix?

## Dataset

The full BRFSS 2022 annual file (`LLCP2022.XPT`, ~1.16 GB, 445,132
respondents x 328 columns) was downloaded from the CDC
(`scripts/fetch_brfss2022.py`) and reduced to a 24-column subset
(`data/brfss2022_subset.csv`) containing only the variables used here.

- **Outcome**: `heart_disease`, derived from `_MICHD` -- CDC's computed
  "ever told you had coronary heart disease or myocardial infarction"
  indicator.
- **Protected attribute**: `SEXVAR` (1 = Male, 2 = Female).
- **Case-mix covariates**: age, race/ethnicity, income, education
  (demographics); diabetes, stroke, asthma, COPD, kidney disease,
  arthritis, depression, difficulty walking, BMI, general health,
  physical/mental health days (comorbidities); physical activity, heavy
  drinking, smoking (behavioral); health coverage, cost barriers (access).

See `src/datasets.load_brfss` docstring for the full recoding details.
Missing values (BRFSS's "don't know" / "refused" codes) are recoded to NaN
and then median-imputed -- a documented simplification, not hidden.

As in dataset 2, we use `threshold="prevalence"` since heart disease
prevalence here is ~9%.

In [ ]:
import sys, os
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
RESULTS_DIR = ROOT / "results"
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.datasets import load_brfss
from src.pipeline import run_fairness_analysis
from src import figures as figs

pd.set_option("display.width", 120)

## 1. Load and confirm the data

In [ ]:
d = load_brfss()

df = d["df"]
print("Dataset:", d["name"])
print("Shape:", df.shape)
print("Target column:", d["target_col"], " | Group column:", d["group_col"])
print("Group labels:", d["group_labels"])
print("\nNumber of features:", len(d["feature_cols"]))
print("Any nulls in features?", df[d["feature_cols"]].isnull().sum().sum())

In [ ]:
# Subgroup sizes and positive-case counts BEFORE modeling (TRIPOD+AI standard).
target_col, group_col = d["target_col"], d["group_col"]

for val, label in d["group_labels"].items():
    sub = df[df[group_col] == val]
    n_pos = sub[target_col].sum()
    print(f"{label:8s}: n={len(sub):,}  positives={n_pos:,}  prevalence={n_pos/len(sub):.4f}")

Both subgroups have tens of thousands of positive cases. Unlike dataset 2,
there is a **large** prevalence gap here: men have heart disease at roughly
**1.6x** the rate of women (11.2% vs 7.1%) in this sample. This is the kind
of gap where prevalence adjustment should do substantial work on PPV/NPV --
similar in spirit to dataset 1, but in the opposite direction (men have
*higher* prevalence here, whereas in dataset 1 men also had higher diabetes
prevalence).

## 2. Run the full pipeline

Group A = Male, Group B = Female; gaps are **(Male) - (Female)**.
`threshold="prevalence"`.

In [ ]:
result = run_fairness_analysis(
    df=d["df"],
    feature_cols=d["feature_cols"],
    target_col=d["target_col"],
    group_col=d["group_col"],
    group_a_value=1.0,   # Male
    group_b_value=2.0,   # Female
    group_labels=d["group_labels"],
    covariate_blocks=d["covariate_blocks"],
    threshold="prevalence",
    test_size=0.3,
    random_state=42,
    n_boot=1000,
    verbose=True,
    # Corrected path: preprocessing is fitted on training rows only,
    # and (where a clustering identifier exists) the split, selection
    # CV, calibration folds and bootstrap are patient-grouped.
    cluster_ids=None,
    preprocess_spec=d["preprocess_spec"],
)

## 3. Reading the results

**Raw gaps (Step 3).** Men have ~1.6x the heart-disease prevalence of
women. The raw PPV gap is large and positive (the model's positive
predictions are "more often right" for men) -- exactly what we'd expect
purely from the prevalence difference, even before considering any model
behavior. Sensitivity is *slightly* lower for men (`equal_opportunity_diff`
negative), and FNR slightly higher.

**Prevalence adjustment (Step 4).** This is the headline result for this
dataset: the raw PPV gap (~+0.10) attenuates by **~84%** once both groups
are standardized to the same prevalence -- most of the apparent PPV
advantage for men really was a prevalence artifact. NPV attenuates by
~89% similarly. The predicted-positive-rate gap, however, gets *larger*
(more negative) after adjustment -- a reminder that prevalence adjustment
doesn't uniformly shrink every prevalence-sensitive metric in the same
direction.

**Bootstrap CIs (Step 5).** All raw gaps are significant given the large
sample size. After prevalence adjustment, the PPV gap shrinks from
~+0.10 to ~+0.016 but remains significant -- a *residual* PPV gap, much
smaller than the raw gap but not zero. The sensitivity/FNR gap
(~-0.02 / +0.02) is small but significant and prevalence-invariant.

**Calibration adjustment (Step 6).** As in the other datasets, the
equal-sensitivity thresholds collapse `equal_opportunity_diff` to ~0 by
construction, while `predicted_positive_rate`/`disparate_impact_ratio`
move in compensation.

**Case-mix waterfall (Step 7).** Demographics and comorbidities each
*increase* the magnitude of the Male/Female log-odds association with
heart disease (a negative `pct_of_raw_gap_explained` means the gap got
*bigger*, not smaller) -- i.e. once age, race, income, education, diabetes,
BMI, etc. are accounted for, the sex difference in heart-disease risk is
even larger than the raw comparison suggests. This is a genuinely different
case-mix pattern than dataset 2.

## 4. Figures

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

fig1 = figs.plot_calibration_curve(
    result["y_test"], result["prob"], result["g_test"],
    group_labels={1.0: "Male", 2.0: "Female"},
    title="Calibration by sex (calibrated model) -- BRFSS 2022 heart disease",
)
fig1.savefig(RESULTS_DIR / "03_brfss_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for key in ("ppv", "npv", "predicted_positive_rate"):
    raw = result["bootstrap_raw"][key]
    adjd = result["bootstrap_prevalence_adjusted"][key]
    gap_dict = {
        "Raw": (raw["point"], raw["ci_low"], raw["ci_high"]),
        "Prevalence-adjusted": (adjd["point"], adjd["ci_low"], adjd["ci_high"]),
    }
    fig = figs.plot_gap_bars(
        gap_dict,
        title=f"{key}: raw vs. prevalence-adjusted gap (Male - Female)",
        ylabel="Gap (Male - Female)",
    )
    fig.savefig(RESULTS_DIR / f"03_brfss_gap_{key}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
fig_wf = figs.plot_case_mix_waterfall(
    result["case_mix"],
    title="BRFSS 2022: case-mix waterfall (Male vs Female heart-disease gap)",
)
fig_wf.savefig(RESULTS_DIR / "03_brfss_case_mix_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Save summary table for cross-dataset pooling

In [ ]:
rows = []
for key in ("ppv", "npv", "predicted_positive_rate"):
    raw = result["bootstrap_raw"][key]
    adjd = result["bootstrap_prevalence_adjusted"][key]
    pa = result["prevalence_adjustment"][key]
    rows.append({
        "dataset": "brfss2022",
        "metric": key,
        "raw_gap": raw["point"],
        "raw_ci_low": raw["ci_low"],
        "raw_ci_high": raw["ci_high"],
        "prevalence_adjusted_gap": adjd["point"],
        "prevalence_adjusted_ci_low": adjd["ci_low"],
        "prevalence_adjusted_ci_high": adjd["ci_high"],
        "attenuation_pct": pa["attenuation_pct"],
    })

for key in ("sensitivity", "fnr"):
    raw = result["bootstrap_raw"][key]
    rows.append({
        "dataset": "brfss2022",
        "metric": key,
        "raw_gap": raw["point"],
        "raw_ci_low": raw["ci_low"],
        "raw_ci_high": raw["ci_high"],
        "prevalence_adjusted_gap": np.nan,
        "prevalence_adjusted_ci_low": np.nan,
        "prevalence_adjusted_ci_high": np.nan,
        "attenuation_pct": np.nan,
    })

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS_DIR / "03_brfss_summary.csv", index=False)
summary

In [ ]:
# Frozen held-out outputs.
#
# Persist the row-level held-out predictions, the overall and per-subgroup
# discrimination/calibration table, and the fitted preprocessing + estimator +
# calibration objects. Nothing here changes the analysis: every value written
# is read from `result`, which was produced above. These artifacts let any
# later evaluation be answered without refitting a model.
from src import frozen_outputs as fo

frozen_manifest = fo.write_comparison_artifacts(
    result,
    slug="sex_brfss",
    dataset=d["name"],
    comparison_type="sex",
    comparison="Male vs Female",
    results_dir=RESULTS_DIR,
    cluster_ids=None,
)
print("frozen predictions :", frozen_manifest["predictions"])
print("frozen performance :", frozen_manifest["performance"])
print("serialized objects :", frozen_manifest["model"]["status"])
